In [1]:
# Imports
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Define input file paths
project_root = Path.cwd().parent.parent

forecast_dir = project_root / "data" / "forecasting"

population_forecast_path = forecast_dir / "SAP_Forecast_2136_Cleaned.csv"

area_zones_dir = project_root / "data" / "area_zones_data"

area_zones_path = area_zones_dir / "Vic_Area_Zones.shp"

In [5]:
# Load the population forecast and the SA2 boundary data
population_data = pd.read_csv(population_forecast_path)
area_zones = gpd.read_file(area_zones_path)

DataSourceError: /home/aydin-khan/Documents/coding/applied-data-science-projects/real-estate-industry-project/data_to_use/area_zones_data/Vic_Area_Zones.shp: No such file or directory

In [ ]:
# Define the five-year age groups used in the forecast column names

age_groups = [
    "0-4",
    "5-9",
    "10-14",
    "15-19",
    "20-24",
    "25-29",
    "30-34",
    "35-39",
    "40-44",
    "45-49",
    "50-54",
    "55-59",
    "60-64",
    "65-69",
    "70-74",
    "75-79",
    "80-84",
    "85+",
]

In [ ]:
# Calculate the total population of each SA2 in the base year (2026)
# and the forecast year (2036)

population_data["population_2026"] = population_data[
    [f"Persons_2026_{age}" for age in age_groups]
].sum(axis=1)

population_data["population_2036"] = population_data[
    [f"Persons_2036_{age}" for age in age_groups]
].sum(axis=1)

# Calculate the percentage population growth between 2026 and 2036 for each SA2

population_data["population_growth_percent"] = (
    (population_data["population_2036"] - population_data["population_2026"])
    / population_data["population_2026"]
    * 100
)

In [ ]:
# Growth is undefined where the 2026 population is zero (division by zero),
# so set it to missing
population_data.loc[
    population_data["population_2026"] == 0, "population_growth_percent"] = np.nan

In [ ]:
# Standardise the SA2 code columns as stripped strings so the two datasets
# can be joined reliably

population_data["SA2_Code"] = population_data["SA2_Code"].astype(str).str.strip()

area_zones["SA2_CODE21"] = area_zones["SA2_CODE21"].astype(str).str.strip()

In [ ]:
# Join the population figures to the SA2 boundaries

population_map = area_zones.merge(
    population_data[
        [
            "SA2_Code",
            "population_2026",
            "population_2036",
            "population_growth_percent",
        ]
    ],
    left_on="SA2_CODE21",
    right_on="SA2_Code",
    how="left",
)

In [ ]:
# Map 1: Forecast population growth by SA2 across Victoria (default colour scale)

fig, ax = plt.subplots(figsize=(12, 12))

population_map.plot(
    column="population_growth_percent",
    cmap="YlOrRd",
    linewidth=0.2,
    edgecolor="grey",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    legend_kwds={"label": "Population Growth (%)", "shrink": 0.7},
    ax=ax,
)

ax.set_title(
    "Forecast Population Growth by SA2, Victoria\n2026–2036",
    fontsize=18,
    fontweight="bold",
)
ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Shared settings for the diverging maps

# Diverging colour map: red = population decline, yellow = no change, green = growth
growth_cmap = LinearSegmentedColormap.from_list(
    "red_yellow_green",
    ["#b2182b", "#fff2a8", "#1b7837"],
)

# Growth values above this cap are shown at the top of the colour scale
# (for visualisation only; the underlying data are not modified)
growth_cap_percent = 50

population_map["population_growth_plot"] = population_map[
    "population_growth_percent"
].clip(upper=growth_cap_percent)

# Centre the colour scale on 0% so decline and growth are visually distinct
growth_norm = TwoSlopeNorm(
    vmin=population_map["population_growth_percent"].min(),
    vcenter=0,
    vmax=growth_cap_percent,
)


def format_colourbar_tick(value, _position):
    """Label the capped end of the colour bar as '50+' rather than '50'."""
    if np.isclose(value, growth_cap_percent):
        return f"{growth_cap_percent}+"
    return f"{int(value)}"


def plot_growth_map(gdf, title, xlim=None, ylim=None):
    """Plot capped population growth by SA2 on a diverging colour scale.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        SA2 boundaries with a ``population_growth_plot`` column.
    title : str
        Map title.
    xlim, ylim : tuple of float, optional
        Axis limits (longitude, latitude) used to zoom to a region.
    """
    fig, ax = plt.subplots(figsize=(12, 12))

    gdf.plot(
        column="population_growth_plot",
        cmap=growth_cmap,
        norm=growth_norm,
        linewidth=0.2,
        edgecolor="grey",
        legend=True,
        missing_kwds={"color": "lightgrey", "label": "No data"},
        legend_kwds={"label": "Population Growth (%)", "shrink": 0.7},
        ax=ax,
    )

    # The colour bar is the last axes added to the figure
    colourbar_ax = fig.axes[-1]
    colourbar_ax.yaxis.set_major_formatter(FuncFormatter(format_colourbar_tick))

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    ax.set_title(title, fontsize=18, fontweight="bold")
    ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Map 2: Forecast population growth by SA2, zoomed to Melbourne
plot_growth_map(
    population_map,
    title="Forecast Population Growth by SA2\nInner Melbourne 2026–2036",
    xlim=(144.50, 145.65),
    ylim=(-38.60, -37.50),
)

In [ ]:
# Map 3: Forecast population growth by SA2 across Victoria
plot_growth_map(
    population_map,
    title="Forecast Population Growth by SA2\nVictoria 2026–2036",
)

In [ ]:
# Summary statistics for population growth across all SA2s
population_map["population_growth_percent"].describe()

In [ ]:
# The 50 fastest-growing SA2s
(
    population_map[
        [
            "SA2_NAME21",
            "population_2026",
            "population_2036",
            "population_growth_percent",
        ]
    ]
    .sort_values("population_growth_percent", ascending=False)
    .head(50)
)